###### Step 1: Imports and configuration

In [0]:
#Install Vector Search client
%pip install databricks-vectorsearch
dbutils.library.restartPython()

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.vector_search.client import VectorSearchClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

w = WorkspaceClient()
vsc = VectorSearchClient()
vsc.list_endpoints()
vsc = VectorSearchClient(disable_notice=True)

VECTOR_SEARCH_ENDPOINT_NAME = "telco-vector-search-endpoint"
INDEX_NAME =  "dbw_agentic_ai_dev.telco_ai.customer_note_embeddings_index"
EMBEDDING_MODEL = "databricks-gte-large-en"
LLM_MODEL = "databricks-meta-llama-3-1-8b-instruct"

index = vsc.get_index(
    endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
    index_name=INDEX_NAME
)


###### Step 2: Create Tool #1 (Vector Search Tool)

In [0]:
def search_customer_notes(question, num_results=3):

    response = w.serving_endpoints.query(
        name=EMBEDDING_MODEL,
        input=[question]
    )

    question_embedding = [
        float(x)
        for x in response.data[0].embedding
    ]

    results = index.similarity_search(
        query_vector=question_embedding,
        columns=["customer_id", "note"],
        num_results=num_results
    )

    return results

######Step 3: Create Tool #2 (SQL Tool)
This tool will answer --> How many notes do we have?
without using: Embedding, Vector Search, LLM. Much faster.

In [0]:
def count_customer_notes():

    count = spark.sql("""
        SELECT COUNT(*)
        FROM dbw_agentic_ai_dev.telco_ai.customer_notes
    """).collect()[0][0]

    return count

###### Step 4: Create First Agent Router

In [0]:
def agent(question):

    question_lower = question.lower()

    if "how many" in question_lower:

        count = count_customer_notes()

        return f"There are {count} customer notes."

    else:

        return search_customer_notes(question)


The Agent is making a decision.
Question
      ↓
Decision
      ↓
Tool

In [0]:
agent("How many customer notes are there?")

Agent execution:
Question
    ↓
Contains "how many"
    ↓
SQL Tool
    ↓
SELECT COUNT(*)
    ↓
8

In [0]:
agent("Why are customers cancelling service?")

Agent execution:
Question
    ↓
Not a count question
    ↓
Vector Search Tool
    ↓
Similarity Search
    ↓
Top 3 Notes

Above is sharing raw results but we need final answer. so instead of just returning results we will use prompt and get the final answer from LLM

Now:
Question
   ↓
Agent
   ↓
Tool
   ↓
Raw Results

After:
Question
   ↓
Agent
   ↓
Tool
   ↓
LLM
   ↓
Answer

###### with LLM

In [0]:
def search_customer_notes(question, num_results=3):
    response = w.serving_endpoints.query(
        name=EMBEDDING_MODEL,
        input=[question]
    )

    question_embedding = [
        float(x)
        for x in response.data[0].embedding
    ]

    results = index.similarity_search(
        query_vector=question_embedding,
        columns=["customer_id", "note"],
        num_results=num_results
    )

    rows = results["result"]["data_array"]

    context_lines = []

    for customer_id, note, score in rows:
        context_lines.append(
            f"Customer {int(customer_id)}: {note}"
        )

    context = "\n".join(context_lines)

    return context, rows

In [0]:
def generate_answer(prompt):
    response = w.serving_endpoints.query(
        name=LLM_MODEL,
        messages=[
            ChatMessage(
                role=ChatMessageRole.USER,
                content=prompt
            )
        ],
        max_tokens=300,
        temperature=0.0
    )

    return response.choices[0].message.content

In [0]:
def agent(question):
    question_lower = question.lower()

    if "how many" in question_lower:
        count = count_customer_notes()
        return f"There are {count} customer notes."

    else:
        context, rows = search_customer_notes(question)

        prompt = f"""
You are a helpful telco customer support analyst.

Answer the user's question using only the context below.

Context:
{context}

Question:
{question}

Answer:
"""

        answer = generate_answer(prompt)
        return answer

In [0]:
agent("How many customer notes are there?")

In [0]:
agent("Why are customers cancelling service?")